# Cvičení: Analýza hlavních komponent (PCA) – Diamanty

## Zadání cvičení:
1. Načtěte dataset `diamonds_preprocessed.csv` do prostředí Google Colab / Jupyter.
2. Importujte všechny potřebné metody z knihoven `Pandas` a `Scikit-learn`.
3. Vytvořte DataFrame na základě souboru `diamonds_preprocessed.csv`.
4. Přiřaďte nezávislé proměnné do proměnné `X` a závislou proměnnou (cenu `price`) do proměnné `y`.
5. Na základě příslušného atributu třídy `PCA` vypočítejte a zaznamenejte skóre vysvětleného rozptylu (variance score) pro různý počet hlavních komponent (od 1 po celkový počet nezávislých proměnných v datasetu). Výsledek zapište do slovníku (`dict`), kde klíčem bude počet komponent a hodnotou bude skóre rozptylu (použijte cyklus `for`).
6. Vyberte takový počet komponent, který vysvětluje **minimálně 80 % a maximálně 90 %** původního rozptylu v datech.
7. Vytvořte hlavní komponenty na základě zvoleného počtu a sestavte nový DataFrame `diamonds_pca_df` obsahující určené hlavní komponenty a závislou proměnnou `price`.

### Krok 1 a 2: Import potřebných knihoven

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

### Krok 3: Načtení datasetu `diamonds_preprocessed.csv`

In [ ]:
# Načtení dat
df = pd.read_csv('diamonds_preprocessed.csv')
print(f"Rozměry datasetu: {df.shape}")
df.head()

### Krok 4: Rozdělení na matici příznaků `X` a cílovou proměnnou `y`
Závislou proměnnou je cena diamantu (`price`), nezávislými veličinami jsou všechny ostatní sloupce (fyzikální rozměry a kódované parametry kvality).

In [ ]:
X = df.drop(columns=['price'])
y = df['price']

print(f"Nezávislé proměnné (X): {X.shape[1]} sloupců -> {list(X.columns)}")
print(f"Závislá proměnná (y): {y.name} ({len(y)} záznamů)")

### Krok 5: Výpočet vysvětleného rozptylu pro 1 až N komponent pomocí cyklu `for`
Využijeme atribut `pca.explained_variance_ratio_`, jehož součet (`np.sum`) udává celkový podíl zachyceného rozptylu.

In [ ]:
# Standardizace je pro PCA klíčová (přednáškový standard ze slidu 9)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Slovník pro uložení skóre rozptylu
variance_scores = {}

for n in range(1, X.shape[1] + 1):
    pca = PCA(n_components=n)
    pca.fit(X_scaled)
    variance_scores[n] = float(np.sum(pca.explained_variance_ratio_))

# Zobrazení výsledného slovníku
for n, score in variance_scores.items():
    print(f"Počet komponent: {n} -> Vysvětlený rozptyl: {score:.4f} ({score*100:.2f} %)")

### Krok 6: Výběr počtu komponent splňujících kritérium (80 % až 90 % rozptylu)
Hledáme počet komponent $k$, pro který platí: $0.80 \le \text{score} \le 0.90$.

In [ ]:
selected_n = None
for n, score in variance_scores.items():
    if 0.80 <= score <= 0.90:
        selected_n = n
        print(f"Kritériu (80 % až 90 %) vyhovuje {n} komponent s rozptylem {score*100:.2f} %.")

print(f"Vybraný počet komponent: {selected_n}")

### Krok 7: Vytvoření komponent a sestavení nového DataFrame `diamonds_pca_df`
Fitujeme PCA s `n_components=4` a transformujeme standardizovaná data do nové ortogonální báze.

In [ ]:
# Fit a transformace na zvolený počet komponent
pca_final = PCA(n_components=selected_n)
X_pca = pca_final.fit_transform(X_scaled)

# Vytvoření nového DataFrame s komponentami a cílovou proměnnou price
col_names = [f"PC{i+1}" for i in range(selected_n)]
diamonds_pca_df = pd.DataFrame(X_pca, columns=col_names)
diamonds_pca_df['price'] = y.values

print("Výsledný DataFrame diamonds_pca_df:")
diamonds_pca_df.head(10)

### Kontrola a vizualizace rozptylu (Scree Plot)
Podívejme se, jak jednotlivé komponenty přispívají k celkovému rozptylu.

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar([f"PC{i+1}" for i in range(selected_n)], pca_final.explained_variance_ratio_ * 100, color='royalblue', alpha=0.8, label='Jednotlivé PC')
plt.plot([f"PC{i+1}" for i in range(selected_n)], np.cumsum(pca_final.explained_variance_ratio_) * 100, color='darkorange', marker='o', linewidth=2, label='Kumulativní rozptyl')
plt.axhline(80, color='red', linestyle='--', alpha=0.7, label='80 % hranice')
plt.axhline(90, color='green', linestyle='--', alpha=0.7, label='90 % hranice')
plt.title('Vysvětlený rozptyl vybraných 4 komponent (Diamonds PCA)')
plt.ylabel('Podíl rozptylu (%)')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()